# SGLang baseline: Qwen2.5-3B-Instruct on a Colab T4

Third engine for the vLLM vs. naive HF benchmark. Runs `benchmark_sglang.py` with the same
prompts, batch sizes (1/4/8/16/32), `max_new_tokens=256`, `temperature=0.7`, `top_p=1.0` and
seed 42 as `benchmark_vllm.py`, then re-runs `analyze_results.py` so SGLang shows up as a third
series next to the committed vLLM and baseline results.

**GPU.** Pick *Runtime > Change runtime type > T4 GPU*. Current SGLang needs sm80+ (Ampere), and
the T4 is sm75, so this notebook pins **sglang 0.4.6.post5**, the newest release whose kernels
are still built for sm75 on torch 2.6 / triton 3.2 (see `requirements-sglang.txt`). If it fails
on the T4, use the L4 fallback cell at the bottom.

**Separate environment.** SGLang and vLLM pin different torch versions, so this notebook
installs only the SGLang stack. Every benchmark runs as `!python ...` in a fresh subprocess,
so no runtime restart is needed after the install.

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv

In [ ]:
!git clone https://github.com/HarshShroff/vllm-inference-benchmark.git
%cd vllm-inference-benchmark
# Until the SGLang PR is merged the files live on the sglang-baseline branch.
!git checkout sglang-baseline 2>/dev/null || echo "sglang-baseline branch not found, using main"

## Install (pinned for T4 / sm75)
`uv` resolves the pinned stack in a couple of minutes; plain `pip install -r requirements-sglang.txt`
also works but is slower. The FlashInfer wheel comes from FlashInfer's own index (cu124 / torch 2.6).

In [ ]:
!pip uninstall -y -q torchaudio torchvision
!pip install -q uv
!uv pip install --system -q -r requirements-sglang.txt
!python -c "import sglang, torch; print('sglang', sglang.__version__, '| torch', torch.__version__, '| cuda ok', torch.cuda.is_available())"

## Smoke test
One short generation before the full sweep, to fail fast if the kernels do not load on this GPU.
On sm75 the benchmark script switches to `--attention-backend triton --sampling-backend pytorch`
automatically; the smoke test does the same.

In [ ]:
%%writefile smoke_sglang.py
import sglang as sgl, pynvml
pynvml.nvmlInit()
cc = pynvml.nvmlDeviceGetCudaComputeCapability(pynvml.nvmlDeviceGetHandleByIndex(0))
kw = dict(attention_backend="triton", sampling_backend="pytorch") if tuple(cc) < (8, 0) else {}
if __name__ == "__main__":
    llm = sgl.Engine(model_path="Qwen/Qwen2.5-3B-Instruct", dtype="float16",
                     mem_fraction_static=0.85, random_seed=42, **kw)
    out = llm.generate(prompt="What is the time complexity of binary search?",
                       sampling_params={"temperature": 0.7, "top_p": 1.0, "max_new_tokens": 32})
    print(out["text"]); print(out["meta_info"]["completion_tokens"], "tokens")
    llm.shutdown()

In [ ]:
!python smoke_sglang.py

## Full sweep
15 runs (3 length buckets x 5 batch sizes). Each run is a non-streaming batched `Engine.generate`
for throughput, then a separate streaming pass over the same batch for TTFB. Writes
`results/sglang_results.json` with per-request token counts.

In [ ]:
!python benchmark_sglang.py --model Qwen/Qwen2.5-3B-Instruct --output results/sglang_results.json

## Compare against the committed vLLM / baseline results
The vLLM and baseline JSONs in `results/` came from an earlier T4 session. Same GPU model, same
prompts and settings, but a different Colab session, so treat small differences with care.
Only rows marked `Equal tokens = yes` are an equal-work throughput comparison.

In [ ]:
!pip install -q pandas matplotlib
!python analyze_results.py --vllm results/vllm_results.json --baseline results/baseline_results.json --sglang results/sglang_results.json --summary-out results/SUMMARY_3engine.md

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open("results/SUMMARY_3engine.md").read()))
display(Image("plots/throughput_vs_batch.png"))
display(Image("plots/memory_vs_batch.png"))

In [ ]:
import json
d = json.load(open("results/sglang_results.json"))
print({k: v for k, v in d.items() if k != "runs"})
for r in d["runs"]:
    print(r["length_bucket"], r["batch_size"], r["total_tokens_generated"], r["throughput_tokens_per_sec"])

In [ ]:
!zip -r sglang_benchmark_results.zip results/ plots/
from google.colab import files
files.download("sglang_benchmark_results.zip")

## Fallback: if the T4 run fails
If the install or smoke test fails on the T4 (for example `no kernel image is available for
execution on the device`), switch the runtime to an **L4** (sm89) and install a current SGLang
instead of the pinned one, then re-run the smoke test and sweep:

```
!uv pip install --system -q "sglang[all]" nvidia-ml-py pandas matplotlib
```

Numbers from an L4 are **not** comparable with the committed T4 vLLM/baseline results; re-run
`benchmark_vllm.py` and `benchmark_baseline.py` on the same L4 (separate environment) before
putting them in one table.